# Notebook 3: The Knowledge Graph in Amazon Neptune

In notebooks 01–02, we built and validated the ontology locally using rdflib and pyshacl. That works for development, but in production the ontology needs to be:

- **Persistent** - survives restarts, shared across consumers
- **Queryable** - agents, dashboards, and services can discover the schema at runtime
- **Governed** - IAM authentication, audit logs, access control

This is where Amazon Neptune comes in. Neptune isn't just a data store - it's where the **ontology becomes a runtime resource that agents depend on**.

### Why store the ontology in Neptune?

Ontop needs the ontology as a flat file to compile its OBDA mappings. But agents need something different - they need to **introspect the ontology dynamically**:

- "What classes exist in this domain?"
- "What properties does a Claim have?"
- "What's the range of `forPolicy`?"

When the ontology is loaded into Neptune, these become SPARQL queries. The labels and comments we added in Notebook 01 aren't just documentation - they're **machine-readable context that agents consume at runtime**.

### The ontology lives in three places

| Location | Purpose |
|----------|---------|
| **Git/S3** | Source of truth - version-controlled artifact |
| **Ontop** (flat file) | Mapping contract - compiles OBDA mappings at boot |
| **Neptune** (triples) | Discovery layer - agents query the schema at runtime |

All three get the same file. CI/CD keeps them in sync.


---
### Setup


In [ ]:
%graph_notebook_config --store-to NOTEBOOK_CONFIG --silent

In [ ]:
from workshop import *
import json
configure(neptune_config=json.loads(NOTEBOOK_CONFIG))

---
## Step 1: Load the Ontology into Neptune

First, we load the ontology - making the schema queryable.


In [ ]:
# Load the ontology into Neptune
ontology_ttl = (NOTEBOOK_DIR / "ontology.ttl").read_text()
status = neptune_update(f"INSERT DATA {{ {ontology_ttl} }}")
print(f"Loaded ontology.ttl: HTTP {status}")

## Step 2: Query the Ontology

Now an agent can discover the domain model by querying Neptune directly. This is the key difference from a flat file - the schema is a **live, queryable resource**.


In [ ]:
%%sparql

# What classes exist in this domain?
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?class ?label ?comment
WHERE {
    ?class a owl:Class ;
           rdfs:label ?label ;
           rdfs:comment ?comment .
}
ORDER BY ?label


The same ontology structure, now as a graph. Switch to the **Graph** tab to see how the classes connect through object properties - this is the domain model at a glance:


In [ ]:
%%sparql

# Visualize the full ontology structure as a graph
# Switch to the Graph tab to explore classes and their relationships
PREFIX : <http://example.org/insurance#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

CONSTRUCT {
    ?prop rdfs:domain ?domain .
    ?prop rdfs:range ?range .
}
WHERE {
    ?prop a owl:ObjectProperty ;
          rdfs:domain ?domain ;
          rdfs:range ?range .
}


An agent investigating a specific class can drill into its properties. This query shows everything the ontology says about what a Claim *is* - its attributes and relationships:


In [ ]:
%%sparql

# What properties does a Claim have?
PREFIX : <http://example.org/insurance#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?property ?label ?range
WHERE {
    ?property rdfs:domain :Claim ;
              rdfs:label ?label .
    OPTIONAL { ?property rdfs:range ?range }
}
ORDER BY ?label


Switch to the **Graph** tab in the results below to see the Claim class and its properties visualized as a graph - this is the structure an agent would discover when introspecting the ontology:


In [ ]:
%%sparql

# Visualize the Claim class and its properties as a graph
# Switch to the Graph tab to see the ontology structure
PREFIX : <http://example.org/insurance#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

CONSTRUCT {
    ?property rdfs:domain :Claim .
    ?property rdfs:range ?range .
    ?property a ?propType .
}
WHERE {
    ?property rdfs:domain :Claim ;
              a ?propType .
    OPTIONAL { ?property rdfs:range ?range }
    FILTER (?propType IN (owl:DatatypeProperty, owl:ObjectProperty))
}


An agent building a SPARQL query can now introspect the schema first - it doesn't need the ontology hardcoded in its prompt. In Notebook 05, we'll see Bedrock do exactly this.

---
## Step 3: Load Instance Data

Now we load the claim and policy data alongside the ontology.


In [ ]:
# Load instance data into Neptune
data_ttl = (NOTEBOOK_DIR / "data.ttl").read_text()
status = neptune_update(f"INSERT DATA {{ {data_ttl} }}")
print(f"Loaded data.ttl: HTTP {status}")

## Step 4: Query the Knowledge Graph

With both ontology and data loaded, we can query claims just as we did locally in Notebook 02 - but now against a persistent, IAM-authenticated graph database.


In [ ]:
%%sparql

PREFIX : <http://example.org/insurance#>
SELECT ?claimId ?claimAmount ?policyId ?incidentDate
WHERE {
    ?claim a :Claim ;
           :claimId ?claimId ;
           :claimAmount ?claimAmount ;
           :incidentDate ?incidentDate ;
           :forPolicy ?policy .
    ?policy :policyId ?policyId .
}
ORDER BY ?claimId

The same 5 claims, the same SPARQL - but now served by Neptune with IAM auth, audit logging, and serverless scaling. The ontology and data are persistent and accessible to any authorized consumer.

→ Proceed to **04-Virtual-Knowledge-Graph**
